# Audit Cube to Bowl before training

Grade [`triton7777/cube_to_bowl_so100`](https://huggingface.co/datasets/triton7777/cube_to_bowl_so100), 10 SO-100 pick-and-place episodes, with Kalanos and see what it flags.
CPU only, no API key, nothing is published. Choose **Runtime → Run all**.

## Install

Kalanos with video checks, and two helpers, at pinned versions.

In [ ]:
# @title Install Kalanos
%pip install -q --no-warn-conflicts \
    "kalanos[numeric,video,hf]==0.10.1" huggingface-hub==1.33.0 plotly==7.1.0

## Download the dataset

About 66 MB, pinned to one revision so every run sees the same files.

In [ ]:
# @title Download the dataset
from pathlib import Path

from huggingface_hub import snapshot_download


DATASET_ID = "triton7777/cube_to_bowl_so100"
# A fixed revision keeps the files, and so every result below, the same on each run.
REVISION = "9f490c4e624034eb5c6e961e65767abb9ab7c391"

DATASET = Path(
    snapshot_download(
        repo_id=DATASET_ID,
        repo_type="dataset",
        revision=REVISION,
        local_dir="cube_to_bowl_so100",
    )
)

## Grade it

The CLI prints a report card and saves the full report as JSON: 9 of 10 episodes pass and 1 needs review.
It uses the `vision-imitation-v1` profile at the full tier, which adds camera checks.

In [ ]:
# @title Grade it with the Kalanos CLI
# `--verbosity error` hides warnings about file formats this dataset doesn't use.
# Piping through `cat` stops the CLI from animating a progress spinner in the cell,
# and `--color` keeps the card's colours through the pipe.
!kalanos --verbosity error grade {DATASET} --profile vision-imitation-v1 --tier full --report audit.json --color 2>&1 | cat

## Explore the report in Python

One row per episode. Review is a result, not an error: Kalanos found something worth a look, and the decision is yours.

In [ ]:
# @title Load the report into a table
import polars as pl

import kalanos


loaded = kalanos.load_report("audit.json")
# `load_report` also reads reports from older schemas, which have no episodes.
# Assigning after the check gives `report` that narrower type in later cells too.
assert isinstance(loaded, kalanos.Report)
report = loaded

rows = [
    {"episode": e.id, "decision": e.eligibility.status.value}
    for e in report.episodes
    # Every episode in a saved report has a decision; this only tells the type checker.
    if e.eligibility is not None
]

# Wide enough to show the episode number at the end of each id.
pl.Config.set_fmt_str_lengths(50)
pl.DataFrame(rows).sort("decision", "episode", descending=[True, False])

## What Kalanos flagged

Episode 000008 goes to review because its gripper camera is blurrier than that camera's other episodes.
Compare sharpness within one camera, not across cameras: it depends on the lens and the scene.
Kalanos calls this a heuristic: a reason to look, not proof of a fault.

In [ ]:
# @title Chart camera sharpness by episode
import plotly.express as px


# Grey recedes and orange carries the story; both clear 3:1 on the card surface.
NO_FLAG_COLOR, FLAGGED_COLOR = "#94938d", "#eb6834"
SURFACE, GRID, INK, MUTED = "#fcfcfb", "#e1e0d9", "#2b2b29", "#6b6a65"
CAMERA_PREFIX = "observation.images."

# The camera readings Kalanos flagged, found in the report as finding → episode and camera.
flagged = {
    (f.episode_id, f.source_field)
    for f in report.findings
    if f.metric_id == "vision.blur_vs_camera"
}

# Every camera reading, found as episode → stream → metric.
points = []
for e in report.episodes:
    for s in e.streams:
        sharpness = s.metrics.get("sharpness_score")
        # Only camera streams carry a sharpness score; a missing one is never plotted as zero.
        if sharpness is None or sharpness.value is None:
            continue
        camera = s.source_field or "camera"
        points.append(
            {
                "episode": int(e.id.rsplit("_", 1)[1]),
                "camera": camera.removeprefix(CAMERA_PREFIX),
                "sharpness": sharpness.value,
                "blur_flag": "Flagged"
                if (e.id, s.source_field) in flagged
                else "No blur flag",
            }
        )
sharpness_by_episode = pl.DataFrame(points)

fig = px.scatter(
    sharpness_by_episode,
    x="episode",
    y="sharpness",
    color="blur_flag",
    color_discrete_map={"No blur flag": NO_FLAG_COLOR, "Flagged": FLAGGED_COLOR},
    category_orders={"blur_flag": ["Flagged", "No blur flag"]},
    facet_col="camera",
    custom_data=["episode", "blur_flag"],
    labels={
        "episode": "<b>DATASET</b> · episode",
        "sharpness": "<b>KALANOS</b> · sharpness score",
        "blur_flag": "Camera blur flag",
    },
    height=480,
)
fig.update_traces(
    marker={"size": 15, "line": {"color": SURFACE, "width": 2}},
    hovertemplate="<b>Episode %{customdata[0]}</b><br>sharpness %{y:.3f}"
    "<br>Camera blur flag: %{customdata[1]}<extra></extra>",
)
# Each camera gets its own scale; the panel title is the camera's name.
fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('=')[-1]}</b>"))
fig.update_yaxes(matches=None, showticklabels=True)
axis = {"gridcolor": GRID, "linecolor": "#c3c2b7", "zeroline": False, "ticks": ""}
fig.update_xaxes(axis, dtick=1, title_standoff=16)
fig.update_yaxes(axis, title_standoff=12)

n_review = len(flagged)
plural = "reading" if n_review == 1 else "readings"
fig.update_layout(
    font={
        "family": "system-ui, -apple-system, 'Segoe UI', Roboto, sans-serif",
        "size": 15,
        "color": INK,
    },
    paper_bgcolor=SURFACE,
    plot_bgcolor=SURFACE,
    margin={"t": 130, "r": 30},
    title={
        "text": f"Kalanos sends {n_review} camera {plural} to review for blur",
        "font": {"size": 22},
        "subtitle": {
            "text": "One dot per episode and camera. Each panel has its own scale.",
            "font": {"color": MUTED, "size": 15},
        },
    },
    legend={"title_text": "Camera blur flag"},
)
fig.show()

# The measurement behind the flag, from the finding itself.
for f in report.findings:
    if f.metric_id == "vision.blur_vs_camera":
        median = (f.evidence or {}).get("camera_median")
        baseline = "" if median is None else f" against a camera median of {median:.1f}"
        print(
            f"{f.episode_id} · {f.source_field}: blur {f.value:.1f} {f.unit}{baseline}"
        )

## Download the reports

The report as an HTML page and as JSON, zipped.

In [ ]:
# @title Write the HTML report and download both
import zipfile

from IPython.display import FileLink, display


# The same grade again, written as an HTML page.
# `tail` keeps the card's last lines: the file written, or the error if it failed.
!kalanos --verbosity error grade {DATASET} --profile vision-imitation-v1 --tier full --report audit.html 2>&1 | tail -n 2

with zipfile.ZipFile("kalanos-reports.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write("audit.json")
    archive.write("audit.html")

try:
    # Only Colab has this module; it sends the file to the browser.
    from google.colab import files  # pyright: ignore[reportMissingImports]
except ImportError:
    display(FileLink("kalanos-reports.zip"))
else:
    files.download("kalanos-reports.zip")

**Continue with Kalanos:** [Browse dataset reports](https://hub.kalanos.ai) · [Use the grader locally](https://github.com/KalanosAI/kalanos) · [Attach an audit to a training run](https://github.com/KalanosAI/kalanos-tracking)